<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/AudioCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets

In [2]:
# 1. Install the specialized directory splitting library
!pip install -q split_folders


In [3]:
import torch
import torch.nn as nn
from datasets import load_dataset
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import splitfolders
import librosa
import librosa.display
import torch
from torch.utils.data import Dataset,DataLoader
import numpy as np
import torch.optim as optim
from tqdm import tqdm

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [6]:
#Large Dataset
!kaggle datasets download -d chrisfilo/urbansound8k

Dataset URL: https://www.kaggle.com/datasets/chrisfilo/urbansound8k
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 5.61G/5.61G [02:43<00:00, 37.0MB/s]



In [7]:
!unzip -q urbansound8k.zip -d ./data_folder/

In [8]:
# Define your input and output folders exactly as we mapped them out!
input_folder = '/content/data_folder' #'/content/urban_sounds_small/urban_sounds_small'
output_folder = '/content/urban_sounds8k_output'#'/content/urban_sounds_split'

# Execute the balanced physical split
splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(0.7, 0.3)
)

print("\n[SUCCESS] Folders split perfectly into train and test directories!")



Copying files: 0 files [00:00, ? files/s]
Copying files: 9 files [00:00, 85.61 files/s]
Copying files: 18 files [00:00, 87.48 files/s]
Copying files: 30 files [00:00, 99.56 files/s]
Copying files: 44 files [00:00, 112.86 files/s]
Copying files: 56 files [00:00, 92.40 files/s] 
Copying files: 66 files [00:00, 94.20 files/s]
Copying files: 79 files [00:00, 102.49 files/s]
Copying files: 94 files [00:00, 115.25 files/s]
Copying files: 113 files [00:01, 136.72 files/s]
Copying files: 130 files [00:01, 134.40 files/s]
Copying files: 144 files [00:01, 133.72 files/s]
Copying files: 158 files [00:01, 134.29 files/s]
Copying files: 175 files [00:01, 144.03 files/s]
Copying files: 191 files [00:01, 147.58 files/s]
Copying files: 206 files [00:01, 141.23 files/s]
Copying files: 223 files [00:01, 146.93 files/s]
Copying files: 238 files [00:01, 139.42 files/s]
Copying files: 253 files [00:02, 137.43 files/s]
Copying files: 268 files [00:02, 139.21 files/s]
Copying files: 282 files [00:02, 127.96


[SUCCESS] Folders split perfectly into train and test directories!


In [15]:
!rm -rf /content/preprocessed

In [16]:
%%writefile preprocess.py

import os
import torch
import librosa
import numpy as np
from tqdm import tqdm

def preprocess_split(source_root,save_root,target_sr=16000,duration=3,n_mels=128,augment=False):
    os.makedirs(save_root, exist_ok=True)
    num_samples = target_sr * duration
    class_names = sorted([f for f in os.listdir(source_root) if os.path.isdir(os.path.join(source_root,f))])
    for class_name in class_names:
      class_folder = os.path.join(source_root,class_name)
      save_class_folder = os.path.join(save_root,class_name)
      os.makedirs(save_class_folder,exist_ok=True)
      files = os.listdir(class_folder)

      for file_name in tqdm(files,desc=class_name):
        if not file_name.endswith('.wav'):
          continue

        file_path = os.path.join(class_folder,file_name)
        audio, sr = librosa.load(file_path,sr=target_sr)

        if augment:
          #Noise
          if np.random.rand() < 0.5:
            noise = np.random.randn(len(audio))
            audio = audio + 0.005 * noise

          # Time shift
          if np.random.rand() < 0.5:
            shift = np.random.randint(int(0.1 * target_sr))
            audio = np.roll(audio, shift)

        # trim
        if len(audio) > num_samples:
          audio = audio[:num_samples]
        else:
          padding = num_samples - len(audio)
          audio = np.pad( audio,(0,padding))

        mel = librosa.feature.melspectrogram(y=audio,sr=sr,n_mels=n_mels)

        mel_db = librosa.power_to_db(mel,ref=np.max)

        mel_tensor = torch.tensor(mel_db,dtype=torch.float32)

        mel_tensor = mel_tensor.unsqueeze(0)

        save_path = os.path.join(save_class_folder,file_name.replace('.wav','.pt'))

        torch.save(mel_tensor,save_path)



if __name__ == "__main__":
  preprocess_split('/content/urban_sounds8k_output/train','/content/preprocessed/train',augment=True)
  preprocess_split('/content/urban_sounds8k_output/val','/content/preprocessed/val')

Overwriting preprocess.py


In [17]:
!python preprocess.py

fold1: 100% 611/611 [00:21<00:00, 27.80it/s]
fold10: 100% 585/585 [00:19<00:00, 30.12it/s]
fold2: 100% 621/621 [00:20<00:00, 29.96it/s]
fold3: 100% 647/647 [00:19<00:00, 33.63it/s]
fold4: 100% 693/693 [00:23<00:00, 30.00it/s]
fold5: 100% 655/655 [00:19<00:00, 33.15it/s]
fold6: 100% 576/576 [00:19<00:00, 29.76it/s]
fold7: 100% 586/586 [00:18<00:00, 31.82it/s]
fold8: 100% 564/564 [00:17<00:00, 32.03it/s]
fold9: 100% 571/571 [00:20<00:00, 28.25it/s]
fold1: 100% 262/262 [00:06<00:00, 39.95it/s]
fold10: 100% 252/252 [00:08<00:00, 30.41it/s]
fold2: 100% 267/267 [00:06<00:00, 40.07it/s]
fold3: 100% 278/278 [00:08<00:00, 31.80it/s]
fold4: 100% 297/297 [00:09<00:00, 31.06it/s]
fold5: 100% 281/281 [00:06<00:00, 40.74it/s]
fold6: 100% 247/247 [00:07<00:00, 31.05it/s]
fold7: 100% 252/252 [00:06<00:00, 39.74it/s]
fold8: 100% 242/242 [00:08<00:00, 29.83it/s]
fold9: 100% 245/245 [00:06<00:00, 39.11it/s]


In [11]:
class MelSpectrogramDataset(Dataset):
  def __init__(self,rootFolder) -> None:
    super().__init__()
    self.filePaths = []
    self.labels = []

    #Get sorted folder names to freeze class indexing
    self.class_names = sorted([f for f in os.listdir(rootFolder) if os.path.isdir(os.path.join(rootFolder,f))])

    for class_idx,class_name in enumerate(self.class_names):
      class_folder = os.path.join(rootFolder,class_name)
      for file_name in os.listdir(class_folder):
        if file_name.endswith('.pt'):
          self.filePaths.append(os.path.join(class_folder,file_name))
          self.labels.append(class_idx)

  def __len__(self):
    return len(self.filePaths)

  def __getitem__(self, idx):
    mel_tensor = torch.load(self.filePaths[idx])
    return mel_tensor,self.labels[idx]


In [18]:
train_data = MelSpectrogramDataset(rootFolder='/content/preprocessed/train')

val_data = MelSpectrogramDataset(rootFolder='/content/preprocessed/val')

In [19]:
print(len(train_data))

6109


In [20]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,num_workers=0,pin_memory=True)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,num_workers=0,pin_memory=True)

In [21]:
class CNN(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(1,32,3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.Conv2d(32,32,3),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64,64,3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,3,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128,128,3),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        # nn.Conv2d(128,256,3,padding=1),
        # nn.BatchNorm2d(256),
        # nn.ReLU(),
        # nn.Conv2d(256,256,3),
        # nn.BatchNorm2d(256),
        # nn.ReLU(),
        # nn.MaxPool2d(2,2)
    )

    self.flatten = nn.Flatten()

    self.fc = nn.Sequential(
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128,10)
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)
    return x

In [22]:
model = CNN().to(device)

In [23]:
optimizer = optim.AdamW(model.parameters(),lr=1e-5,weight_decay=1e-3)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

/tmp/ipykernel_4824/2678194489.py:3: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler('cuda')


In [ ]:
epochs = 30
for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  Preloading_train = tqdm(train_loader,desc=f'Training_Epoch: {epoch+1}:')
  for images,label in Preloading_train:
    images = images.to(device)
    label = label.to(device)

    optimizer.zero_grad()

    with torch.amp.autocast('cuda'):
      outputs = model(images)
      loss = loss_fn(outputs,label)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    train_loss += loss.item()
    pred = torch.argmax(outputs, dim=1)
    train_correct += (pred == label).sum().item()
    train_total += label.size(0)
  train_accuracy = train_correct / train_total
  train_average_loss = train_loss / len(train_loader)


  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    Preloading_val = tqdm(val_loader,desc=f'Validation_Epoch: {epoch+1}:')
    for images,label in Preloading_val:
      images = images.to(device)
      label = label.to(device)

      with torch.amp.autocast('cuda'):
        outputs = model(images)
        loss = loss_fn(outputs,label)

      val_loss += loss.item()
      pred = torch.argmax(outputs, dim=1)
      val_correct += (pred == label).sum().item()
      val_total += label.size(0)
    val_accuracy = val_correct / val_total
    val_average_loss = val_loss / len(val_loader)

  print('='*50)
  print(f'Epoch: {epoch+1}')
  print(f'Training_Accuracy: {train_accuracy} | Training_loss: {train_average_loss}')
  print(f'Validation_Accuracy: {val_accuracy} | Validation_loss: {val_average_loss}')
  print('='*50)


Training_Epoch: 1::   0%|          | 0/191 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_4824/2637276189.py:15: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):
Validation_Epoch: 1::   0%|          | 0/82 [00:00<?, ?it/s]/tmp/ipykernel_4824/2637276189.py:42: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):
Validation_Epoch: 1:: 100%|██████████| 82/82 [01:14<00:00,  1.11it/s]


Epoch: 1
Training_Accuracy: 0.11491242429202815 | Training_loss: 2.3091615806699424
Validation_Accuracy: 0.15707205489897064 | Validation_loss: 2.278443967423788


Validation_Epoch: 2:: 100%|██████████| 82/82 [01:16<00:00,  1.07it/s]


Epoch: 2
Training_Accuracy: 0.153543951546898 | Training_loss: 2.266614526978338
Validation_Accuracy: 0.19329012581014107 | Validation_loss: 2.255499278626791


Validation_Epoch: 3:: 100%|██████████| 82/82 [01:22<00:00,  1.00s/it]


...

Validation_Epoch: 5:: 100%|██████████| 82/82 [01:18<00:00,  1.05it/s]


Epoch: 5
Training_Accuracy: 0.23195285644131608 | Training_loss: 2.1551975785749744
Validation_Accuracy: 0.2512390392680137 | Validation_loss: 2.163248067948876


Validation_Epoch: 6:: 100%|██████████| 82/82 [01:17<00:00,  1.06it/s]


Epoch: 6
Training_Accuracy: 0.2619086593550499 | Training_loss: 2.108307149397765
Validation_Accuracy: 0.272969881814716 | Validation_loss: 2.120947496193211


Validation_Epoch: 7::  34%|███▍      | 28/82 [00:28<00:53,  1.02it/s]

In [ ]:
# !rm -rf /content/preprocessed